In [0]:
---Confirm row count and basic shape
SELECT COUNT(*) AS total_rows
FROM blviewer.channel.viewership_analysis;

In [0]:
---Check the DateID field, range and format
SELECT 
  MIN(DateID) AS earliest,
  MAX(DateID) AS latest,
  COUNT(DISTINCT DateID) AS distinct_dates
FROM blviewer.channel.viewership_analysis;

In [0]:
WITH date_range AS (
  SELECT explode(sequence(
    to_date('2020-11-01'), 
    to_date('2021-04-16'), 
    interval 1 day
  )) AS calendar_date
),
actual_dates AS (
  SELECT DISTINCT to_date(CAST(DateID AS STRING), 'yyyyMMdd') AS actual_date
  FROM blviewer.channel.viewership_analysis
)
SELECT calendar_date
FROM date_range
LEFT JOIN actual_dates ON date_range.calendar_date = actual_dates.actual_date
WHERE actual_dates.actual_date IS NULL;

**Observation: Missing Date in Viewership Data**

During date range validation of blviewer.channel.viewership_analysis, the dataset was found to span 166 distinct dates between 2020-11-01 and 2021-04-16 (an inclusive range that should contain 167 calendar days). A gap-check against a full calendar sequence identified 2020-12-01 as the single missing date — no viewership records exist for this day.

Impact: Any time-series analysis, day-over-day comparisons, or rolling/weekly aggregations spanning early December 2020 may show an artificial dip or null value on this date due to the absence of source data, not an actual drop in viewership.

Status: Root cause not yet confirmed — could be a genuine gap in the source file (Viewership Analysis.xlsx) or an issue introduced during ingestion. Recommend verifying against the original source before treating this as a confirmed data gap.

In [0]:
---parsing DateID into a real date and check for failures
SELECT COUNT(*) AS unparseable_rows
FROM blviewer.channel.viewership_analysis
WHERE to_date(CAST(DateID AS STRING), 'yyyyMMdd') IS NULL;

In [0]:
---Checking for duplicate rows
SELECT DateID, CustomerID, TotalTimeWatched, Platform, PlayEventType, VideoTitle, COUNT(*) AS occurrences
FROM blviewer.channel.viewership_analysis
GROUP BY DateID, CustomerID, TotalTimeWatched, Platform, PlayEventType, VideoTitle
HAVING COUNT(*) > 1
ORDER BY occurrences DESC;

%md
### 📌 Observation: Duplicate Rows
Duplicate-check found **5,731 groups** of rows sharing identical `DateID, CustomerID, TotalTimeWatched, Platform, PlayEventType, VideoTitle` values. Verified against the source file (`Viewership Analysis.xlsx`) — duplicates exist in the **original data**, not introduced during ingestion. No timestamp/session ID column exists to distinguish repeat entries as separate events.

⚠️ Impact: Any count of viewing events or sum of watch time will be inflated if duplicates are not handled. Recommend deduplication (e.g. `SELECT DISTINCT`) before analysis, unless repeat plays are intentionally meaningful to the use case.

In [0]:
%sql
-- Create a deduplicated version of the table, keeping the original raw table untouched
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_clean AS
SELECT DISTINCT *
FROM blviewer.channel.viewership_analysis;

In [0]:
%sql
SELECT 
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis) AS raw_rows,
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_clean) AS clean_rows;

%md
### 📌 Observation: Deduplication Applied
Created `viewership_analysis_clean` using `SELECT DISTINCT` on the raw table. This removed **9,873 duplicate rows** (118,534 → 108,661), representing exact matches across `DateID, CustomerID, TotalTimeWatched, Platform, PlayEventType, VideoTitle`. Original raw table (`viewership_analysis`) preserved untouched for audit purposes.

⚠️ Assumption: With no timestamp/session ID in the source data, duplicates were treated as logging errors rather than genuine repeat views. This assumption should be noted in any downstream analysis or report.

In [0]:
SELECT*
FROM blviewer.channel.viewership_analysis_clean
LIMIT 10;

In [0]:
SELECT MIN(TotalTimeWatched), MAX(TotalTimeWatched), AVG(TotalTimeWatched)
FROM blviewer.channel.viewership_analysis_clean;

In [0]:
---Format consistency
SELECT LENGTH(CustomerID) AS id_length, COUNT(*) AS count
FROM blviewer.channel.viewership_analysis_clean
GROUP BY LENGTH(CustomerID)
ORDER BY id_length;

In [0]:
---Null or blank check
SELECT COUNT(*) AS null_or_blank_ids
FROM blviewer.channel.viewership_analysis_clean
WHERE CustomerID IS NULL OR TRIM(CustomerID) = '';

In [0]:
---Distinct customer count
SELECT COUNT(DISTINCT CustomerID) AS unique_customers
FROM blviewer.channel.viewership_analysis_clean;

In [0]:
%sql
---Case sensitivity / trailing whitespace check
SELECT COUNT(DISTINCT CustomerID) AS raw_distinct,
       COUNT(DISTINCT LOWER(TRIM(CustomerID))) AS normalized_distinct
FROM blviewer.channel.viewership_analysis_clean;

%md
### 📌 Observation: CustomerID Validated
`CustomerID` passed all checks:
- **Format**: All 108,661 rows have a consistent 28-character ID.
- **Completeness**: 0 null or blank values.
- **Uniqueness**: 929 unique customers, with raw and case/whitespace-normalized distinct counts matching exactly (929 = 929) — no hidden duplicates from casing or trailing spaces.

✅ No cleaning required. Column is ready for customer-level analysis (e.g., unique viewers, viewing frequency, segmentation).

In [0]:
%sql
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_final AS
SELECT *,
       ROUND(TotalTimeWatched / 60, 2) AS total_minutes_watched,
       ROUND(TotalTimeWatched / 3600, 2) AS total_hours_watched
FROM blviewer.channel.viewership_analysis_clean;

In [0]:
%sql
SELECT *
FROM blviewer.channel.viewership_analysis_final
LIMIT 10;

In [0]:
%sql
SELECT 
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_clean) AS clean_rows,
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_final) AS final_rows;

%md
### 📌 Observation: Final Table Created — Data Validation Complete
Created `viewership_analysis_final` from `viewership_analysis_clean`, adding two derived columns: `total_minutes_watched` and `total_hours_watched` (both derived from `TotalTimeWatched`, confirmed to be in seconds). Row count verified unchanged (108,661 → 108,661) — no data lost or duplicated in this step.

**Summary of full validation process:**
| Check | Result |
|---|---|
| Total raw rows | 118,534 |
| Date range | 2020-11-01 to 2021-04-16 (166/167 days — missing 2020-12-01) |
| Unparseable dates | 0 |
| Duplicate rows removed | 9,873 (118,534 → 108,661) |
| CustomerID format/nulls/uniqueness | All clean (929 unique customers) |
| TotalTimeWatched unit | Confirmed seconds; minute/hour columns added |
| Final table row count | 108,661 |

✅ `viewership_analysis_final` is ready for downstream analysis (trends, platform breakdown, top content, customer segmentation).

In [0]:
---Daily trend (total hours watched + session count per day)
SELECT 
  to_date(CAST(DateID AS STRING), 'yyyyMMdd') AS view_date,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  ROUND(AVG(total_minutes_watched), 2) AS avg_minutes_per_session
FROM blviewer.channel.viewership_analysis_final
GROUP BY view_date
ORDER BY view_date;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
---Weekly trend (smoother view, useful for spotting patterns)
SELECT 
  date_trunc('week', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS week_start,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  COUNT(DISTINCT CustomerID) AS active_viewers
FROM blviewer.channel.viewership_analysis_final
GROUP BY week_start
ORDER BY week_start;

In [0]:
%sql
---Monthly trend (highest level view)
SELECT 
  date_trunc('month', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS month_start,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  COUNT(DISTINCT CustomerID) AS active_viewers
FROM blviewer.channel.viewership_analysis_final
GROUP BY month_start
ORDER BY month_start;

In [0]:
%sql
---day-of-week breakdown
SELECT 
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  ROUND(AVG(total_minutes_watched), 2) AS avg_minutes_per_session
FROM blviewer.channel.viewership_analysis_final
GROUP BY day_of_week
ORDER BY total_sessions DESC;

In [0]:
%sql
SELECT PlayEventType, VideoTitle, COUNT(*) AS sessions
FROM blviewer.channel.viewership_analysis_final
WHERE date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') = 'Thursday'
GROUP BY PlayEventType, VideoTitle
ORDER BY sessions DESC
LIMIT 10;

In [0]:
%sql
---"Unknown" VideoTitle under LiveTV
SELECT COUNT(*) AS unknown_title_count,
       ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_final), 2) AS pct_of_total
FROM blviewer.channel.viewership_analysis_final
WHERE VideoTitle = 'Unknown' OR VideoTitle IS NULL;

In [0]:
%sql
---day-of-week concentration
SELECT 
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  COUNT(*) AS unknown_sessions
FROM blviewer.channel.viewership_analysis_final
WHERE VideoTitle = 'Unknown' OR VideoTitle IS NULL
GROUP BY day_of_week
ORDER BY unknown_sessions DESC;

In [0]:
%sql
SELECT 
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched
FROM blviewer.channel.viewership_analysis_final
WHERE VideoTitle != 'Unknown' AND VideoTitle IS NOT NULL
GROUP BY day_of_week
ORDER BY total_sessions DESC;

---Observation: Weekly Viewership Pattern Confirmed

Day-of-week analysis (including full dataset) shows Sunday and Saturday as peak viewing days, with Thursday consistently ranking 3rd — even after excluding the 31.19% of rows with unlabeled ("Unknown") VideoTitle. This confirms Thursday's elevated viewership is a genuine pattern, not a data artifact.

Top named content on Thursdays includes local drama series (Suidooster, 7de Laan, Arendsvlei, Binnelanders, The Block Australia) — suggesting a Thursday episode release schedule may be driving increased engagement.

✅ Weekly seasonality (weekend + Thursday peak) is a validated finding for trend and scheduling analysis.

In [0]:
%sql
---Top content overall
SELECT 
  VideoTitle,
  PlayEventType,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  COUNT(DISTINCT CustomerID) AS unique_viewers
FROM blviewer.channel.viewership_analysis_final
WHERE VideoTitle != 'Unknown' AND VideoTitle IS NOT NULL
GROUP BY VideoTitle, PlayEventType
ORDER BY total_hours_watched DESC
LIMIT 20;

---Key observations:

***News/current affairs LiveTV dominates the top spots — "South Africa Tonight," "The South African Morning," "Morning Live," "Today," "Lunchtime," "All Angles" are all news/talk formats. This makes sense as regular, habitual daily viewing rather than one-off entertainment.
****"The South African Morning" has the highest hours (713.23) despite fewer sessions (744) than "South Africa Tonight" (1,151 sessions, 696.23 hours) — meaning its average session length is much longer. Worth checking: 713.23 hrs / 744 sessions ≈ 57.5 min/session vs. 696.23/1151 ≈ 36.3 min/session. That's a notably "stickier" show.
***"The Block Australia" stands out — high hours (301.96) and sessions (497) but only 17 unique viewers. That means a small, dedicated audience is watching a LOT — roughly 29 sessions per viewer on average. This is a classic "superfan" pattern versus the broad-reach news shows.
***Local South African content (Suidooster, Getroud Met Rugby) sits mid-table — consistent with steady, loyal niche audiences rather than mass reach.

This gives you a nice narrative: broad daily habitual viewing (news) generates the most total volume, while niche entertainment content (Block Australia, Masterchef) drives deep engagement from a small loyal base.

In [0]:
---Platform breakdown
SELECT 
  Platform,
  COUNT(*) AS total_sessions,
  ROUND(SUM(total_hours_watched), 2) AS total_hours_watched,
  COUNT(DISTINCT CustomerID) AS unique_viewers,
  ROUND(AVG(total_minutes_watched), 2) AS avg_minutes_per_session
FROM blviewer.channel.viewership_analysis_final
GROUP BY Platform
ORDER BY total_hours_watched DESC;

In [0]:
%sql
---customer segmentation
WITH customer_activity AS (
  SELECT 
    CustomerID,
    COUNT(*) AS total_sessions,
    ROUND(SUM(total_hours_watched), 2) AS total_hours_watched
  FROM blviewer.channel.viewership_analysis_final
  GROUP BY CustomerID
)
SELECT 
  CASE 
    WHEN total_sessions >= 200 THEN 'Heavy (200+ sessions)'
    WHEN total_sessions >= 50 THEN 'Regular (50-199 sessions)'
    WHEN total_sessions >= 10 THEN 'Occasional (10-49 sessions)'
    ELSE 'Light (<10 sessions)'
  END AS viewer_segment,
  COUNT(*) AS num_customers,
  ROUND(AVG(total_hours_watched), 2) AS avg_hours_per_customer
FROM customer_activity
GROUP BY viewer_segment
ORDER BY avg_hours_per_customer DESC;

In [0]:
%sql
---Heavy viewers actually watch vs. Light viewers
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_segmented AS
SELECT *,
  COUNT(*) OVER (PARTITION BY CustomerID) AS customer_total_sessions,
  CASE 
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 200 THEN 'Heavy (200+ sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 50 THEN 'Regular (50-199 sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 10 THEN 'Occasional (10-49 sessions)'
    ELSE 'Light (<10 sessions)'
  END AS viewer_segment
FROM blviewer.channel.viewership_analysis_final;

In [0]:
%sql
WITH ranked_content AS (
  SELECT 
    viewer_segment,
    VideoTitle,
    COUNT(*) AS sessions,
    ROW_NUMBER() OVER (PARTITION BY viewer_segment ORDER BY COUNT(*) DESC) AS rnk
  FROM blviewer.channel.viewership_analysis_segmented
  WHERE viewer_segment IN ('Heavy (200+ sessions)', 'Light (<10 sessions)')
    AND VideoTitle != 'Unknown' AND VideoTitle IS NOT NULL
  GROUP BY viewer_segment, VideoTitle
)
SELECT viewer_segment, VideoTitle, sessions
FROM ranked_content
WHERE rnk <= 10
ORDER BY viewer_segment, sessions DESC;

In [0]:
%sql
---Adding columns for analysis
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_enriched AS
SELECT 
  *,
  
  -- Day of week (Monday, Tuesday, etc.)
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  
  -- Weekend flag
  CASE 
    WHEN date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') IN ('Saturday', 'Sunday') 
    THEN TRUE
    ELSE FALSE
  END AS is_weekend,
  
  -- Content quality flag (TRUE = has a real title, FALSE = Unknown/missing)
  CASE 
    WHEN VideoTitle = 'Unknown' OR VideoTitle IS NULL THEN FALSE
    ELSE TRUE
  END AS has_known_title,
  
  -- Customer viewer segment (based on total sessions per customer across the dataset)
  CASE 
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 200 THEN 'Heavy (200+ sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 50 THEN 'Regular (50-199 sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 10 THEN 'Occasional (10-49 sessions)'
    ELSE 'Light (<10 sessions)'
  END AS viewer_segment

FROM blviewer.channel.viewership_analysis_final;

In [0]:
%sql
SELECT 
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_final) AS final_rows,
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_enriched) AS enriched_rows;

In [0]:
SELECT*
FROM blviewer.channel.viewership_analysis_enriched;

In [0]:
%sql
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_enriched AS
SELECT 
  *,
  
  -- Day of week (Monday, Tuesday, etc.)
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  
  -- Weekend flag
  CASE 
    WHEN date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') IN ('Saturday', 'Sunday') 
    THEN TRUE
    ELSE FALSE
  END AS is_weekend,
  
  -- Content quality flag
  CASE 
    WHEN VideoTitle = 'Unknown' OR VideoTitle IS NULL THEN FALSE
    ELSE TRUE
  END AS has_known_title,
  
  -- Customer viewer segment
  CASE 
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 200 THEN 'Heavy (200+ sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 50 THEN 'Regular (50-199 sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 10 THEN 'Occasional (10-49 sessions)'
    ELSE 'Light (<10 sessions)'
  END AS viewer_segment,
  
  -- Month start (for monthly rollups)
  date_trunc('month', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS month_start,
  
  -- Week start (for weekly rollups)
  date_trunc('week', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS week_start

FROM blviewer.channel.viewership_analysis_final;

In [0]:
%sql
SELECT 
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_final) AS final_rows,
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_enriched) AS enriched_rows;

%md
### 📌 Observation: Enriched Analysis Table Finalized
`viewership_analysis_enriched` now includes six derived columns for reusable analysis:
- `day_of_week` — calendar day name
- `is_weekend` — boolean flag for Saturday/Sunday
- `has_known_title` — boolean flag (FALSE when VideoTitle is 'Unknown' or NULL, ~31% of rows)
- `viewer_segment` — customer engagement tier (Heavy/Regular/Occasional/Light)
- `month_start` — first day of the month, for monthly rollups
- `week_start` — first day of the week, for weekly rollups

Row count verified unchanged (108,661 → 108,661). This is now the final analysis-ready table for all downstream work — trends (daily/weekly/monthly), top content, platform breakdown, and customer segmentation.

In [0]:
SELECT*
FROM blviewer.channel.viewership_analysis_enriched;

In [0]:
%sql
CREATE OR REPLACE TABLE blviewer.channel.viewership_analysis_enriched AS
SELECT 
  *,
  
  -- Day of week
  date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') AS day_of_week,
  
  -- Weekend flag
  CASE 
    WHEN date_format(to_date(CAST(DateID AS STRING), 'yyyyMMdd'), 'EEEE') IN ('Saturday', 'Sunday') 
    THEN TRUE
    ELSE FALSE
  END AS is_weekend,
  
  -- Content quality flag
  CASE 
    WHEN VideoTitle = 'Unknown' OR VideoTitle IS NULL THEN FALSE
    ELSE TRUE
  END AS has_known_title,
  
  -- Customer viewer segment
  CASE 
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 200 THEN 'Heavy (200+ sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 50 THEN 'Regular (50-199 sessions)'
    WHEN COUNT(*) OVER (PARTITION BY CustomerID) >= 10 THEN 'Occasional (10-49 sessions)'
    ELSE 'Light (<10 sessions)'
  END AS viewer_segment,
  
  -- Clean month_start as plain DATE (no timestamp/timezone)
  CAST(date_trunc('month', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS DATE) AS month_start,
  
  -- Clean week_start as plain DATE
  CAST(date_trunc('week', to_date(CAST(DateID AS STRING), 'yyyyMMdd')) AS DATE) AS week_start

FROM blviewer.channel.viewership_analysis_final;

In [0]:
SELECT*
FROM blviewer.channel.viewership_analysis_enriched;

In [0]:
%sql
SELECT 
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_final) AS final_rows,
  (SELECT COUNT(*) FROM blviewer.channel.viewership_analysis_enriched) AS enriched_rows;